python data_ingestion_to_sql.py

# Exploratory Data Analysis

understanding the dataset how data is present in the database across different tables and if we need an aggregated table to solve the business problem

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns',None)
from data_ingestion_to_sql import connection_engine_to_db,upsert_to_db
import warnings
warnings.filterwarnings('ignore')

In [ ]:
master_engine,main_engine=connection_engine_to_db()
query = "SELECT table_name FROM INFORMATION_SCHEMA.TABLES"
tables = pd.read_sql_query(query, main_engine)
tables

In [ ]:
def import_tables():
    master_engine,main_engine=connection_engine_to_db()
    query = "SELECT table_name FROM INFORMATION_SCHEMA.TABLES"
    tables = pd.read_sql_query(query, main_engine)
    

In [ ]:
'''import sqlalchemy as sa
drop_query = """
ALTER DATABASE marketing_e_commerce 
SET SINGLE_USER 
WITH ROLLBACK IMMEDIATE;
DROP DATABASE marketing_e_commerce;
"""
with master_engine.connect() as conn:
    conn.execution_options(isolation_level="AUTOCOMMIT")
    conn.execute(sa.text(drop_query))'''

In [ ]:
pd.read_sql("select count(*) from campaigns",main_engine)

In [ ]:
for table in tables['table_name']:
        print("-"*50,f"{table}","-"*50)
        print(f"count of records: {pd.read_sql(f"select count(*) as count from {table}",main_engine)['count'].values[0]}")
        display(pd.read_sql(f"select top 5 * from {table} ",main_engine))

creating a master table

In [ ]:
master_table_query='''
with transactionsdetails AS
(
    select *,cast(timestamp as date) as transaction_date from transactions
),

customerdetails as
(
    select * from customers
),

productdetails as
(
    select * from products
),

camapigndetails as
(
    select * from campaigns
),

eventdetails as
(
    select *,cast(timestamp as date) as event_date from events
)

select
ed.event_id,
ed.event_type,
ed.customer_id,
ed.session_id,
ed.product_id,
ed.device_type,
ed.traffic_source,
ed.campaign_id,
ed.page_category,
ed.session_duration_sec,
ed.experiment_group,
ed.event_date,

crd.country,
crd.age,
crd.gender,
crd.loyalty_tier,
crd.acquisition_channel as acquisition_source,

pd.category as product_category,
pd.brand as product_brand,
pd.base_price as product_base_price,

cmd.channel as campaign_source,
cmd.objective as campaign_goal,
cmd.target_segment as target_customers,

td.transaction_id,
td.transaction_date,
td.customer_id as td_customer_id,
td.product_id as td_product_id,
td.quantity,
td.discount_applied,
td.gross_revenue,
td.campaign_id as td_campaign_id


from eventdetails ed 
left join customerdetails crd on ed.customer_id=crd.customer_id 
left join productdetails pd on ed.product_id=pd.product_id
left join camapigndetails cmd on ed.campaign_id=cmd.campaign_id
left join transactionsdetails td on ed.customer_id=td.customer_id and
ed.product_id=td.product_id and ed.event_date=td.transaction_date
'''

df_master=pd.read_sql_query(master_table_query,main_engine)

In [ ]:
df_master.to_csv('master_table.csv')

In [ ]:
df_master

In [ ]:
df_master.isna().sum()

In [ ]:
df_master[df_master['td_customer_id'].isna()]

In [ ]:
df_master.loc[df_master['td_customer_id'].isna(),'td_customer_id']=df_master[df_master['td_customer_id'].isna()]['customer_id']

In [ ]:
df_master.isna().sum()

In [ ]:
df_master[df_master['campaign_source'].isna()]

In [ ]:
df_master.loc[df_master['campaign_source'].isna(),'campaign_source']=df_master[df_master['campaign_source'].isna()]['traffic_source']

In [ ]:
df_master.isna().sum()

In [ ]:
df_master.loc[df_master['td_campaign_id'].isna(),'td_campaign_id']=df_master[df_master['td_campaign_id'].isna()]['campaign_id']
df_master.isna().sum()

In [ ]:
df_master.loc[(df_master['quantity'].isna()),['quantity','discount_applied','gross_revenue']]=0

In [ ]:
df_master.isna().sum()

In [ ]:
df_master[df_master['transaction_id'].isna()]

In [ ]:
df_master.loc[df_master['transaction_id'].isna(),'transaction_id']=-1

In [ ]:
df_master.isna().sum()

In [ ]:
df_master[df_master['transaction_date'].isna()]

In [ ]:
df_master.loc[df_master['transaction_date'].isna(),'transaction_date']='1900-01-01'

In [ ]:
'''for table in tables['table_name']:
    print("-"*50,f"{table}","-"*50)
    print(f"count of records: {pd.read_sql(f"select count(*) as count from {table}",main_engine)['count'].values[0]}")
    display(pd.read_sql(f"select top 5 * from {table} ",main_engine))
    master_table_query=
    with transactionsdetails AS
    (
        select *,cast(timestamp as date) as transaction_date from transactions
    ),

    customerdetails as
    (
        select * from customers
    ),

    productdetails as
    (
        select * from products
    ),

    camapigndetails as
    (
        select * from campaigns
    ),

    eventdetails as
    (
        select *,cast(timestamp as date) as event_date from events
    )

    select
    ed.event_id,
    ed.event_type,
    ed.customer_id,
    ed.session_id,
    ed.product_id,
    ed.device_type,
    ed.traffic_source,
    ed.campaign_id,
    ed.page_category,
    ed.session_duration_sec,
    ed.experiment_group,
    ed.event_date,

    crd.country,
    crd.age,
    crd.gender,
    crd.loyalty_tier,
    crd.acquisition_channel as acquisition_source,

    pd.category as product_category,
    pd.brand as product_brand,
    pd.base_price as product_base_price,

    cmd.channel as campaign_source,
    cmd.objective as campaign_goal,
    cmd.target_segment as target_customers,

    td.transaction_id,
    td.transaction_date,
    td.customer_id as td_customer_id,
    td.product_id as td_product_id,
    td.quantity,
    td.discount_applied,
    td.gross_revenue,
    td.campaign_id as td_campaign_id


    from eventdetails ed 
    left join customerdetails crd on ed.customer_id=crd.customer_id 
    left join productdetails pd on ed.product_id=pd.product_id
    left join camapigndetails cmd on ed.campaign_id=cmd.campaign_id
    left join transactionsdetails td on ed.customer_id=td.customer_id and
    ed.product_id=td.product_id and ed.event_date=td.transaction_date
    

    df_master=pd.read_sql_query(master_table_query,main_engine)

    df_master.loc[df_master['td_customer_id'].isna(),'td_customer_id']=df_master[df_master['td_customer_id'].isna()]['customer_id']

    df_master.loc[df_master['campaign_source'].isna(),'campaign_source']=df_master[df_master['campaign_source'].isna()]['traffic_source']

    df_master.loc[df_master['td_campaign_id'].isna(),'td_campaign_id']=df_master[df_master['td_campaign_id'].isna()]['campaign_id']

    df_master.loc[(df_master['quantity'].isna()),['quantity','discount_applied','gross_revenue']]=0

    df_master.loc[df_master['transaction_id'].isna(),'transaction_id']=-1

    df_master.loc[df_master['transaction_date'].isna(),'transaction_date']='1900-01-01'

    events=df_master['event_type'].unique().tolist()
    for event in events:
        mode_val=df_master[(df_master['device_type'].notna())&(df_master['event_type']==event)]['device_type'].mode()[0]
        mask=(df_master['device_type'].isna())&(df_master['event_type']==event)
        df_master.loc[mask,'device_type']=mode_val

    '''

In [ ]:
df_master.isna().sum()

In [ ]:
df_master[df_master['device_type'].notna()].groupby(['event_type','device_type'])['device_type'].count()

In [ ]:
events=df_master['event_type'].unique().tolist()
for event in events:
    mode_val=df_master[(df_master['device_type'].notna())&(df_master['event_type']==event)]['device_type'].mode()[0]
    mask=(df_master['device_type'].isna())&(df_master['event_type']==event)
    df_master.loc[mask,'device_type']=mode_val

In [ ]:
df_master.isna().sum()

In [ ]:
df_master

In [ ]:
df_master[df_master['campaign_id']==0]['traffic_source'].value_counts()

In [ ]:
df_master[df_master['campaign_id']==0]['traffic_source']

In [ ]:
df_master.loc[df_master['campaign_id']==0,'campaign_goal']=df_master[df_master['campaign_id']==0]['traffic_source']

In [ ]:
df_master.isna().sum()

In [ ]:
df_master[(df_master['product_id'].isna())&(df_master['product_category'].isna())]

In [ ]:
df_master[df_master['product_id'].notna()].groupby(['page_category','event_type'],as_index=False).agg(
    event_count=('event_type','count'),
    page_category_count=('page_category','count')
)

In [ ]:
df_master[df_master['product_id'].isna()].groupby(['page_category','event_type'],as_index=False).agg(
    event_count=('event_type','count'),
    page_category_count=('page_category','count')
)

In [ ]:
df_master.loc[(df_master['product_id'].isna())&(df_master['event_type']=='bounce'),['product_id','product_base_price','td_product_id']]=-1
df_master.loc[(df_master['product_brand'].isna())&(df_master['event_type']=='bounce'),['product_brand','product_category']]=str('customer_bounced')
df_master.loc[(df_master['product_id'].isna())&(df_master['event_type']=='purchase'),['product_id','product_base_price','td_product_id']]=-1
df_master.loc[(df_master['product_brand'].isna())&(df_master['event_type']=='purchase'),['product_brand','product_category',]]=str('untracked')

In [ ]:
df_master.isna().sum()

In [ ]:
df_master[df_master['target_customers'].isna()]

In [ ]:
df_master[df_master['target_customers'].isna()]['campaign_source'].value_counts()

In [ ]:
df_master.loc[df_master['target_customers'].isna(),'target_customers']='organic_customers'

In [ ]:
df_master.isna().sum()

In [ ]:
len(df_master[df_master['transaction_id']==-1])

In [ ]:
df_master.loc[(df_master['transaction_id']==-1)&(df_master['td_product_id'].isna()),'td_product_id']=-1

In [ ]:
df_master.isna().sum()

In [ ]:
df_master.info()

In [ ]:
df_master['event_date']=pd.to_datetime(df_master['event_date'],format='%Y-%m-%d')
df_master['transaction_date']=pd.to_datetime(df_master['transaction_date'],format='%Y-%m-%d')

In [ ]:
cat_cols=df_master.select_dtypes(exclude=['number','datetime']).columns.to_list()

In [ ]:
df_master

In [ ]:
for col in cat_cols:
    df_master[col]=df_master[col].str.lower().astype('object')

In [ ]:
df_master.info()

In [ ]:
df_master

In [ ]:
df_master.sort_values(by=['session_id'],ascending=True)

In [ ]:
df_master['session_type']=pd.cut(df_master['session_duration_sec'],bins=[0,10,60,float(np.inf)],labels=['Bouncer','Browser','Engaged'])

In [ ]:
df_master

In [ ]:
df_master['day_num']=df_master[df_master['transaction_date']!=pd.Timestamp('1900-01-01')]['transaction_date'].dt.day_of_week
df_master.loc[df_master['day_num'].isna(),'day_num']=-1

In [ ]:
df_master

In [ ]:
df_master['is_weekend']=df_master['day_num'].apply(lambda x:int(True) if x in [5,6]  else (-1 if x in [-1] else int(False)))
df_master

In [ ]:
cust_total_gross=df_master.groupby(['customer_id'],as_index=False)['gross_revenue'].sum()
cust_total_gross

In [ ]:
cust_total_gross['customer_id'].unique().tolist()

In [ ]:
cust_total_gross.loc[cust_total_gross['customer_id']==9250,'gross_revenue'].values[0]

In [ ]:
df_master['gross_revenue'].sum()

In [ ]:
if cust_total_gross.loc[cust_total_gross['customer_id']==1,'gross_revenue'].values[0]<=(0.01*(df_master['gross_revenue'].sum())):
    df_master['cust_value']='low_value'
elif (cust_total_gross.loc[cust_total_gross['customer_id']==1,'gross_revenue'].vaues[0]>(0.01*(df_master['gross_revenue'].sum()))) or (cust_total_gross.loc[cust_total_gross['customer_id']==1,'gross_revenue'].values[0]<=(0.05*(df_master['gross_revenue'].sum()))):
    df_master['cust_value']='mid_value'
else:
    df_master['cust_value']='high_value'

In [ ]:
'''def cust_value_scoring(df_master):
    cust_total_gross=df_master.groupby(['customer_id'],as_index=False)['gross_revenue'].sum()
    
    gross_total=df_master['gross_revenue'].sum()

    score_map={}
    
    for id in cust_total_gross['customer_id'].unique().tolist():
        if cust_total_gross.loc[cust_total_gross['customer_id']==id,'gross_revenue'].values[0]<=(0.01*(gross_total)):
            score_map[id]='low_value'
        elif (cust_total_gross.loc[cust_total_gross['customer_id']==id,'gross_revenue'].values[0]>(0.01*(gross_total))) or (cust_total_gross.loc[cust_total_gross['customer_id']==id,'gross_revenue'].values[0]<=(0.05*(gross_total))):
            score_map[id]='mid_value'
        else:
            score_map[id]='high_value'
    df_master['cust_value']=df_master['customer_id'].map(score_map)
    return df_master

df_master=cust_value_scoring(df_master)'''

In [ ]:
cust_total_series = df_master.groupby('customer_id')['gross_revenue'].transform('sum')
cust_total_series

In [ ]:
def cust_value_scoring(df_master):
    cust_total_series = df_master.groupby('customer_id')['gross_revenue'].transform('sum')
    gross_total = df_master['gross_revenue'].sum()

    df_master['cust_value'] = pd.cut(
        cust_total_series, 
        bins=[-float(np.inf), 0.01*gross_total, 0.05*gross_total, float(np.inf)], 
        labels=['low_value', 'mid_value', 'high_value']
    )
    return df_master

df_master=cust_value_scoring(df_master)

In [ ]:
df_master

In [ ]:
def get_cart_abandoner(df:pd.DataFrame)->pd.DataFrame:
    df_master=df
    df_master['is_purchase_event']=df_master['event_type'].apply(lambda x:int(True) if 'purchase' in x else int(False))
    
    purchase_ratio_table=df_master.groupby(['customer_id'],as_index=False).agg(
    purchase_session_sum=('is_purchase_event','sum'),
    session_count=('session_id','nunique')
    )
    purchase_ratio_table['purchase_ratio']=(purchase_ratio_table['purchase_session_sum']/purchase_ratio_table['session_count'])*100
    ratio_lookup=purchase_ratio_table.set_index('customer_id')['purchase_ratio']
    df_master['purchase_ratio']=df_master['customer_id'].map(ratio_lookup)
    df_master['is_cart_abandoner']=df_master['purchase_ratio'].apply(lambda x:int(False) if x>=5 else int(True))

    df_master.drop(columns=['is_purchase_event'],inplace=True)
    
    return df_master,purchase_ratio_table
    

In [ ]:
df_master,purchase_ratio_table=get_cart_abandoner(df=df_master)

In [ ]:
df_master

In [ ]:
df_master.info()

In [ ]:
cat_cols=df_master.select_dtypes(include=['str','object']).columns.to_list()
for col in cat_cols:
    unique_count=df_master[col].nunique()
    static_threshold=200
    if unique_count<=static_threshold:
        df_master[col]=df_master[col].astype('category')
    else:
        df_master[col]=df_master[col].astype('object')

In [ ]:
0.001*2000000000